## Seminar 6: FastSpeech

### План:

##### **FastSpeech**

##### **Montreal Forced Alignment**

**FastSpeech** — это неавторегрессионная модель для синтеза речи, которая генерирует мел-спектрограммы значительно быстрее, чем авторегрессионные модели и при этом сохраняет высокое качество речи.

<center><img src="img/fastspeech.png" width=200></center>

**Основные компоненты:**
1. **Feed-Forward Transformer** 
2. **Length Regulator**
3. **Duration Predictor**

В качестве вокодера мы будем использовать **HiFi-GAN**.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
import os
import json
import re
from collections import OrderedDict
from IPython.display import Audio, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


<a name="hyperparameters"></a>
### Гиперпараметры

В этом классе собраны все конфигурационные параметры модели. Они соответствуют оригинальной архитектуре FastSpeech.

Основные параметры модели:
*   `d_model`: размерность скрытых представлений (384)
*   `encoder_n_layer` / `decoder_n_layer`: количество слоев трансформера (6)
*   `vocab_size`: размер словаря символов

In [2]:
class HP:
    # Model
    vocab_size = 1024
    max_sep_len = 2048
    word_vec_dim = 384
    d_model = 384
    
    # Encoder
    encoder_n_layer = 6
    encoder_head = 2
    encoder_conv1d_filter_size = 1536
    encoder_output_size = 384
    
    # Decoder
    decoder_n_layer = 6
    decoder_head = 2
    decoder_conv1d_filter_size = 1536
    decoder_output_size = 384
    
    # Duration Predictor
    duration_predictor_filter_size = 256
    duration_predictor_kernel_size = 3
    
    # FFT Block
    fft_conv1d_kernel = 3
    fft_conv1d_padding = 1
    
    # Mel
    num_mels = 80
    
    # Common
    dropout = 0.1
    
hp = HP()

<a name="text-processing"></a>
### Обработка текста

Модели TTS не работают с текстом напрямую, им нужны числовые идентификаторы. 
Здесь мы определяем словарь символов, который включает:
- Базовую пунктуацию
- Английский алфавит
- ARPAbet фонем (для более точного произношения)

Функция `text_to_sequence` преобразует входную строку в список индексов.

In [3]:
# Full symbol set with ARPAbet phonemes (from original repository)
_pad = '_'
_punctuation = '!\'(),.:;? '
_special = '-'
_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'

# ARPAbet phonemes (84 phonemes from cmudict.valid_symbols)
_arpabet = [
    'AA', 'AA0', 'AA1', 'AA2', 'AE', 'AE0', 'AE1', 'AE2', 'AH', 'AH0', 'AH1', 'AH2',
    'AO', 'AO0', 'AO1', 'AO2', 'AW', 'AW0', 'AW1', 'AW2', 'AY', 'AY0', 'AY1', 'AY2',
    'B', 'CH', 'D', 'DH', 'EH', 'EH0', 'EH1', 'EH2', 'ER', 'ER0', 'ER1', 'ER2', 'EY',
    'EY0', 'EY1', 'EY2', 'F', 'G', 'HH', 'IH', 'IH0', 'IH1', 'IH2', 'IY', 'IY0', 'IY1',
    'IY2', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OW0', 'OW1', 'OW2', 'OY', 'OY0',
    'OY1', 'OY2', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UH0', 'UH1', 'UH2', 'UW',
    'UW0', 'UW1', 'UW2', 'V', 'W', 'Y', 'Z', 'ZH'
]
_arpabet = ['@' + s for s in _arpabet]  # Add '@' prefix for uniqueness

# Checkpoint trained with 149 symbols (148 from repo + 1 extra, likely '~')
symbols = [_pad] + list(_special) + list(_punctuation) + list(_letters) + _arpabet + ['~']
PAD = symbols.index(_pad)

_symbol_to_id = {s: i for i, s in enumerate(symbols)}

def text_to_sequence(text: str) -> list:
    """Convert text to sequence of symbol IDs (simple character-based)"""
    text = text.lower()
    text = re.sub(r'[^a-z!\'(),.:;? -]', '', text)
    sequence = [_symbol_to_id[c] for c in text if c in _symbol_to_id]
    return sequence

print(f"Vocab size: {len(symbols)}")
print(f"Test sequence: {text_to_sequence('Hello!')}")

Vocab size: 149
Test sequence: [45, 42, 49, 49, 52, 2]


<a name="utils"></a>
### Вспомогательные функции

Здесь определены функции для создания масок и позиционного кодирования.

In [4]:
def get_sinusoid_encoding_table(n_position, d_hid, padding_idx=None):
    def cal_angle(position, hid_idx):
        return position / np.power(10000, 2 * (hid_idx // 2) / d_hid)
    
    def get_posi_angle_vec(position):
        return [cal_angle(position, hid_j) for hid_j in range(d_hid)]
    
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(n_position)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    
    if padding_idx is not None:
        sinusoid_table[padding_idx] = 0.
    
    return torch.FloatTensor(sinusoid_table)

def get_non_pad_mask(seq):
    return seq.ne(PAD).type(torch.float).unsqueeze(-1)

def get_attn_key_pad_mask(seq_k, seq_q):
    len_q = seq_q.size(1)
    padding_mask = seq_k.eq(PAD)
    padding_mask = padding_mask.unsqueeze(1).expand(-1, len_q, -1)
    return padding_mask

def pad_list(inputs, mel_max_length=None):
    """Pad list of tensors"""
    if mel_max_length:
        max_len = mel_max_length
    else:
        max_len = max([inputs[i].size(0) for i in range(len(inputs))])
    
    out_list = []
    for batch in inputs:
        one_batch_padded = F.pad(batch, (0, 0, 0, max_len - batch.size(0)), "constant", 0.0)
        out_list.append(one_batch_padded)
    
    return torch.stack(out_list)

<a name="transformer-components"></a>
### Компоненты Трансформера

Основные блоки архитектуры:
1.  **Scaled Dot-Product Attention**
2.  **Multi-Head Attention**
3.  **Position-wise Feed-Forward**

In [13]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)
        self.softmax = nn.Softmax(dim=2)

    def forward(self, q, k, v, mask=None):
        attn = torch.bmm(q, k.transpose(1, 2))
        attn = attn / self.temperature

        if mask is not None:
            attn = attn.masked_fill(mask, -np.inf)

        attn = self.softmax(attn)
        attn = self.dropout(attn)
        output = torch.bmm(attn, v)

        return output, attn

class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model, d_k, d_v, dropout=0.1):
        super().__init__()

        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k)
        self.w_ks = nn.Linear(d_model, n_head * d_k)
        self.w_vs = nn.Linear(d_model, n_head * d_v)

        self.attention = ScaledDotProductAttention(temperature=np.power(d_k, 0.5))
        self.layer_norm = nn.LayerNorm(d_model)

        self.fc = nn.Linear(n_head * d_v, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head

        sz_b, len_q, _ = q.size()
        sz_b, len_k, _ = k.size()
        sz_b, len_v, _ = v.size()

        residual = q

        q = self.w_qs(q).view(sz_b, len_q, n_head, d_k)
        k = self.w_ks(k).view(sz_b, len_k, n_head, d_k)
        v = self.w_vs(v).view(sz_b, len_v, n_head, d_v)

        q = q.permute(2, 0, 1, 3).contiguous().view(-1, len_q, d_k)
        k = k.permute(2, 0, 1, 3).contiguous().view(-1, len_k, d_k)
        v = v.permute(2, 0, 1, 3).contiguous().view(-1, len_v, d_v)

        if mask is not None:
            mask = mask.repeat(n_head, 1, 1)

        output, attn = self.attention(q, k, v, mask=mask)

        output = output.view(n_head, sz_b, len_q, d_v)
        output = output.permute(1, 2, 0, 3).contiguous().view(sz_b, len_q, -1)

        output = self.dropout(self.fc(output))
        output = self.layer_norm(output + residual)

        return output, attn

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_in, d_hid, kernel_size, dropout=0.1):
        super().__init__()
        
        self.w_1 = nn.Conv1d(
            d_in, d_hid, kernel_size=kernel_size[0], padding=(kernel_size[0] - 1) // 2)
        self.w_2 = nn.Conv1d(
            d_hid, d_in, kernel_size=kernel_size[1], padding=(kernel_size[1] - 1) // 2)
        
        self.layer_norm = nn.LayerNorm(d_in)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        output = x.transpose(1, 2)
        output = self.w_2(F.relu(self.w_1(output)))
        output = output.transpose(1, 2)
        output = self.dropout(output)
        output = self.layer_norm(output + residual)
        
        return output

<a name="fft-block"></a>
### FFT Block

**Feed-Forward Transformer (FFT) Block** — это единичный слой энкодера и декодера.
Он состоит из **Multi-Head Attention** и **Conv1D** (Position-wise Feed-Forward) слоёв. 

В FastSpeech энкодер и декодер состоят из стопки таких блоков (по 6 штук).

<center><img src="img/fft.png" width=300></center>

In [14]:
class FFTBlock(nn.Module):
    def __init__(self, d_model, d_inner, n_head, d_k, d_v, dropout=0.1):
        super().__init__()
        self.slf_attn = MultiHeadAttention(n_head, d_model, d_k, d_v, dropout=dropout)
        self.pos_ffn = PositionwiseFeedForward(
            d_model, d_inner, kernel_size=[hp.fft_conv1d_kernel, hp.fft_conv1d_kernel], dropout=dropout)

    def forward(self, enc_input, non_pad_mask=None, slf_attn_mask=None):
        enc_output, enc_slf_attn = self.slf_attn(
            enc_input, enc_input, enc_input, mask=slf_attn_mask)
        
        if non_pad_mask is not None:
            enc_output = enc_output * non_pad_mask

        enc_output = self.pos_ffn(enc_output)
        
        if non_pad_mask is not None:
            enc_output = enc_output * non_pad_mask

        return enc_output, enc_slf_attn

<a name="base-layers"></a>
### Conv и Linear слои

Здесь мы определяем базовые сверточные и линейные слои с правильной инициализацией весов.

*   `Conv`: Используется в `DurationPredictor`.
*   `ConvNorm`: Обычная свертка без транспонирования, используется в `PostNet`.

In [15]:
class Conv(nn.Module):
    """Conv with internal transpose for DurationPredictor"""
    def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
                 padding=0, dilation=1, bias=True, w_init='linear'):
        super().__init__()
        
        self.conv = nn.Conv1d(in_channels, out_channels,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=bias)
        nn.init.xavier_uniform_(self.conv.weight, gain=nn.init.calculate_gain(w_init))

    def forward(self, x):
        # Transpose input: [batch, seq, channels] -> [batch, channels, seq]
        x = x.contiguous().transpose(1, 2)
        x = self.conv(x)
        x = x.contiguous().transpose(1, 2)
        return x

class ConvNorm(nn.Module):
    """ConvNorm without transpose for PostNet"""
    def __init__(self, in_channels, out_channels, kernel_size=1, stride=1,
                 padding=0, dilation=1, bias=True, w_init='linear'):
        super().__init__()
        
        self.conv = nn.Conv1d(in_channels, out_channels,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=bias)
        nn.init.xavier_uniform_(self.conv.weight, gain=nn.init.calculate_gain(w_init))

    def forward(self, x):
        # Uses standard layout: [batch, channels, seq]
        return self.conv(x)

class Linear(nn.Module):
    def __init__(self, in_dim, out_dim, bias=True, w_init='linear'):
        super().__init__()
        self.linear_layer = nn.Linear(in_dim, out_dim, bias=bias)
        nn.init.xavier_uniform_(self.linear_layer.weight,
                                gain=nn.init.calculate_gain(w_init))

    def forward(self, x):
        return self.linear_layer(x)

<a name="post-net"></a>
### Post-Net

**Post-Net** — это модуль постобработки, который улучшает качество сгенерированной мел-спектрограммы.
Он состоит из 5 слоев одномерных сверток. На вход подается спектрограмма, предсказанная декодером, а PostNet предсказывает "остаток" (residual), который добавляется к исходному предсказанию.
Это помогает модели сделать спектрограмму более детальной.

In [16]:
class PostNet(nn.Module):
    def __init__(self, n_mel_channels=80, postnet_embedding_dim=512,
                 postnet_kernel_size=5, postnet_n_convolutions=5):
        super().__init__()
        self.convolutions = nn.ModuleList()

        self.convolutions.append(
            nn.Sequential(
                ConvNorm(n_mel_channels, postnet_embedding_dim,
                         kernel_size=postnet_kernel_size, stride=1,
                         padding=int((postnet_kernel_size - 1) / 2),
                         dilation=1, w_init='tanh'),
                nn.BatchNorm1d(postnet_embedding_dim))
        )

        for i in range(1, postnet_n_convolutions - 1):
            self.convolutions.append(
                nn.Sequential(
                    ConvNorm(postnet_embedding_dim,
                             postnet_embedding_dim,
                             kernel_size=postnet_kernel_size, stride=1,
                             padding=int((postnet_kernel_size - 1) / 2),
                             dilation=1, w_init='tanh'),
                    nn.BatchNorm1d(postnet_embedding_dim))
            )

        self.convolutions.append(
            nn.Sequential(
                ConvNorm(postnet_embedding_dim, n_mel_channels,
                         kernel_size=postnet_kernel_size, stride=1,
                         padding=int((postnet_kernel_size - 1) / 2),
                         dilation=1, w_init='linear'),
                nn.BatchNorm1d(n_mel_channels))
        )

    def forward(self, x):
        # [batch, seq, channels] -> [batch, channels, seq]
        x = x.contiguous().transpose(1, 2)
        
        for i in range(len(self.convolutions) - 1):
            x = torch.tanh(self.convolutions[i](x))
        
        x = self.convolutions[-1](x)
        
        # Back to [batch, seq, channels]
        x = x.contiguous().transpose(1, 2)
        return x

<a name="duration-predictor"></a>
### Duration Predictor

**Duration Predictor** берет на вход скрытые представления фонем из энкодера и предсказывает, сколько фреймов мел-спектрограммы должна занимать каждая фонема.

<center><img src="img/lr.png" width=300></center>

In [17]:
class DurationPredictor(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_size = hp.d_model
        self.filter_size = hp.duration_predictor_filter_size
        self.kernel = hp.duration_predictor_kernel_size
        self.conv_output_size = hp.duration_predictor_filter_size
        self.dropout = hp.dropout

        self.conv_layer = nn.Sequential(OrderedDict([
            ("conv1d_1", Conv(self.input_size, self.filter_size,
                              kernel_size=self.kernel, padding=1)),
            ("layer_norm_1", nn.LayerNorm(self.filter_size)),
            ("relu_1", nn.ReLU()),
            ("dropout_1", nn.Dropout(self.dropout)),
            ("conv1d_2", Conv(self.filter_size, self.filter_size,
                              kernel_size=self.kernel, padding=1)),
            ("layer_norm_2", nn.LayerNorm(self.filter_size)),
            ("relu_2", nn.ReLU()),
            ("dropout_2", nn.Dropout(self.dropout))
        ]))

        self.linear_layer = Linear(self.conv_output_size, 1)
        self.relu = nn.ReLU()

    def forward(self, encoder_output):
        out = self.conv_layer(encoder_output)
        out = self.linear_layer(out)
        out = self.relu(out)
        out = out.squeeze()
        if not self.training:
            out = out.unsqueeze(0)
        return out

<a name="length-regulator"></a>
### Length Regulator

**Length Regulator** выполняет операцию растягивания последовательности фонем в соответствии с предсказанными длительностями.

Если фонема 'A' должна длиться 3 фрейма, то вектор этой фонемы просто дублируется 3 раза.
Параметр $\alpha$ позволяет управлять скоростью речи: умножая длительности на число, мы можем ускорять или замедлять речь.

<center><img src="img/lr.png" width=300></center>

In [18]:
class LengthRegulator(nn.Module):
    def __init__(self):
        super().__init__()
        self.duration_predictor = DurationPredictor()

    def LR(self, x, duration_predictor_output, alpha=1.0, mel_max_length=None):
        output = []
        for batch, expand_target in zip(x, duration_predictor_output):
            output.append(self.expand(batch, expand_target, alpha))
        
        output = pad_list(output, mel_max_length)
        return output

    def expand(self, batch, predicted, alpha):
        out = []
        for i, vec in enumerate(batch):
            expand_size = predicted[i].item() if hasattr(predicted[i], 'item') else predicted[i]
            # Speed control: alpha > 1 slows down, alpha < 1 speeds up 
            out.append(vec.expand(int(expand_size * alpha), -1))
        out = torch.cat(out, 0)
        return out

    def rounding(self, num):
        return int(num) + 1 if num - int(num) >= 0.5 else int(num)

    def forward(self, x, alpha=1.0, target=None, mel_max_length=None):
        duration_predictor_output = self.duration_predictor(x)

        if self.training:
            output = self.LR(x, target, mel_max_length=mel_max_length)
            return output, duration_predictor_output
        else:
            duration_rounded = [self.rounding(ele) for ele in duration_predictor_output[0]]
            duration_rounded = [duration_rounded]
            output = self.LR(x, duration_rounded, alpha)
            
            mel_pos = torch.stack(
                [torch.Tensor([i + 1 for i in range(output.size(1))])]).long().to(device)
            
            return output, mel_pos

<a name="encoder-decoder"></a>
### Энкодер и Декодер

Модули энкодера и декодера состоят из стека FFT-блоков:
*   **Энкодер** преобразует последовательность символов в скрытые представления.
*   **Декодер** преобразует растянутую последовательность после Length Regulator в мел-спектрограмму.

In [19]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        n_src_vocab = len(symbols)
        len_max_seq = hp.max_sep_len
        n_position = len_max_seq + 1
        d_word_vec = hp.word_vec_dim
        n_layers = hp.encoder_n_layer
        n_head = hp.encoder_head
        d_k = 64
        d_v = 64
        d_model = hp.d_model
        d_inner = hp.encoder_conv1d_filter_size
        dropout = hp.dropout

        self.src_word_emb = nn.Embedding(n_src_vocab, d_word_vec, padding_idx=PAD)
        self.position_enc = nn.Embedding.from_pretrained(
            get_sinusoid_encoding_table(n_position, d_word_vec, padding_idx=0), freeze=True)
        
        self.layer_stack = nn.ModuleList([
            FFTBlock(d_model, d_inner, n_head, d_k, d_v, dropout=dropout)
            for _ in range(n_layers)])

    def forward(self, src_seq, src_pos, return_attns=False):
        enc_slf_attn_list = []
        
        slf_attn_mask = get_attn_key_pad_mask(seq_k=src_seq, seq_q=src_seq)
        non_pad_mask = get_non_pad_mask(src_seq)
        
        enc_output = self.src_word_emb(src_seq) + self.position_enc(src_pos)

        for enc_layer in self.layer_stack:
            enc_output, enc_slf_attn = enc_layer(
                enc_output, non_pad_mask=non_pad_mask, slf_attn_mask=slf_attn_mask)
            if return_attns:
                enc_slf_attn_list += [enc_slf_attn]

        return enc_output, non_pad_mask

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        len_max_seq = hp.max_sep_len
        n_position = len_max_seq + 1
        d_word_vec = hp.word_vec_dim
        n_layers = hp.decoder_n_layer
        n_head = hp.decoder_head
        d_k = 64
        d_v = 64
        d_model = hp.d_model
        d_inner = hp.decoder_conv1d_filter_size
        dropout = hp.dropout

        self.position_enc = nn.Embedding.from_pretrained(
            get_sinusoid_encoding_table(n_position, d_word_vec, padding_idx=0), freeze=True)
        
        self.layer_stack = nn.ModuleList([
            FFTBlock(d_model, d_inner, n_head, d_k, d_v, dropout=dropout)
            for _ in range(n_layers)])

    def forward(self, enc_seq, enc_pos, return_attns=False):
        dec_slf_attn_list = []
        
        slf_attn_mask = get_attn_key_pad_mask(seq_k=enc_pos, seq_q=enc_pos)
        non_pad_mask = get_non_pad_mask(enc_pos)
        
        dec_output = enc_seq + self.position_enc(enc_pos)

        for dec_layer in self.layer_stack:
            dec_output, dec_slf_attn = dec_layer(
                dec_output, non_pad_mask=non_pad_mask, slf_attn_mask=slf_attn_mask)
            if return_attns:
                dec_slf_attn_list += [dec_slf_attn]

        return dec_output

<a name="fastspeech-model"></a>
### FastSpeech

Собираем все компоненты вместе в единый класс `FastSpeech`.

1. `Encoder` кодирует текст.
2. `Length Regulator` предсказывает длительности и растягивает вектор.
3. `Decoder` генерирует черновую мел-спектрограмму.
4. `PostNet` улучшает результат.

In [20]:
class FastSpeech(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.length_regulator = LengthRegulator()
        self.decoder = Decoder()

        self.mel_linear = Linear(hp.decoder_output_size, hp.num_mels)
        self.postnet = PostNet()

    def forward(self, src_seq, src_pos, mel_pos=None, mel_max_length=None,
                length_target=None, alpha=1.0):
        encoder_output, non_pad_mask = self.encoder(src_seq, src_pos)

        if self.training:
            length_regulator_output, duration_predictor_output = self.length_regulator(
                encoder_output, target=length_target, alpha=alpha, mel_max_length=mel_max_length)
            decoder_output = self.decoder(length_regulator_output, mel_pos)

            mel_output = self.mel_linear(decoder_output)
            mel_output_postnet = self.postnet(mel_output) + mel_output

            return mel_output, mel_output_postnet, duration_predictor_output
        else:
            length_regulator_output, decoder_pos = self.length_regulator(
                encoder_output, alpha=alpha)

            decoder_output = self.decoder(length_regulator_output, decoder_pos)

            mel_output = self.mel_linear(decoder_output)
            mel_output_postnet = self.postnet(mel_output) + mel_output

            return mel_output, mel_output_postnet

<a name="load-weights"></a>
### Загрузка предобученных весов

Мы загружаем:
1.  **FastSpeech**
2.  **HiFi-GAN**:

HiFi-GAN загружается непосредственно из GitHub репозитория.

In [21]:
# Create and load FastSpeech
fastspeech_model = FastSpeech().to(device)
fastspeech_model.eval()

checkpoint_path = 'model_new/checkpoint_112000.pth.tar'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint
    
    # Remove 'module.' prefix from DataParallel
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace('module.', '') if k.startswith('module.') else k
        new_state_dict[name] = v
    
    fastspeech_model.load_state_dict(new_state_dict)
    print(f"FastSpeech loaded from {checkpoint_path}")
    print(f"Checkpoint size: {os.path.getsize(checkpoint_path) / (1024*1024):.1f} MB")
else:
    print(f"Checkpoint not found: {checkpoint_path}")

FastSpeech loaded from model_new/checkpoint_112000.pth.tar
Checkpoint size: 576.1 MB


In [22]:
# Clone and load HiFi-GAN
if not os.path.exists('hifi-gan'):
    print("Cloning HiFi-GAN...")
    !git clone https://github.com/jik876/hifi-gan.git
    print("HiFi-GAN cloned")
else:
    print("HiFi-GAN exists")

sys.path.insert(0, os.path.join(os.getcwd(), 'hifi-gan'))

HiFi-GAN exists


In [23]:
from env import AttrDict
from models import Generator

hifigan_checkpoint_path = 'hifi-gan/checkpoints/generator_universal.pth'
hifigan_config_path = 'hifi-gan/config_v1.json'

if os.path.exists(hifigan_checkpoint_path) and os.path.exists(hifigan_config_path):
    with open(hifigan_config_path) as f:
        hifigan_config = json.load(f)
    hifigan_config = AttrDict(hifigan_config)
    
    hifigan_model = Generator(hifigan_config).to(device)
    checkpoint = torch.load(hifigan_checkpoint_path, map_location=device, weights_only=False)
    
    if 'generator' in checkpoint:
        hifigan_model.load_state_dict(checkpoint['generator'])
    else:
        hifigan_model.load_state_dict(checkpoint)
    
    hifigan_model.eval()
    hifigan_model.remove_weight_norm()
    
    print(f"HiFi-GAN loaded from {hifigan_checkpoint_path}")
else:
    print("HiFi-GAN not found")

Removing weight norm...
HiFi-GAN loaded from hifi-gan/checkpoints/generator_universal.pth


/Users/nikita/miniconda3/envs/fastspeech1/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


<a name="inference"></a>
### Пайплайн синтеза речи (Inference)

Функция `synthesize_speech` объединяет все шаги:
1.  Предобработка текста (очистка, токенизация).
2.  Инференс FastSpeech (Текст -> Мел-спектрограмма).
3.  Инференс HiFi-GAN (Мел-спектрограмма -> Аудио).

In [24]:
def synthesize_speech(text: str, alpha: float = 1.0) -> np.ndarray:
    sequence = text_to_sequence(text)
    if len(sequence) == 0:
        print("Empty sequence!")
        return np.array([])
    
    src_seq = torch.LongTensor([sequence]).to(device)
    src_pos = torch.arange(1, len(sequence) + 1).unsqueeze(0).to(device)
    
    print(f"Text: '{text}'")
    
    # FastSpeech inference
    with torch.no_grad():
        mel_output, mel_postnet = fastspeech_model(src_seq, src_pos, alpha=alpha)
    
    # Prepare for HiFi-GAN
    mel = mel_postnet[0].cpu().transpose(0, 1).unsqueeze(0)
    
    # HiFi-GAN vocoding
    with torch.no_grad():
        audio = hifigan_model(mel.to(device))
    
    audio = audio.squeeze().cpu().numpy()
    audio = audio / np.abs(audio).max() * 0.95
    
    display(Audio(audio, rate=22050, autoplay=False))
    
    return audio

<a name="experiments"></a>
### Тестирование

Запустим простой тест, чтобы убедиться, что вся система работает корректно.

In [25]:
audio = synthesize_speech("Hello world! This is FastSpeech with HiFi-GAN.")

Text: 'Hello world! This is FastSpeech with HiFi-GAN.'


### Генерация вашего текста

Теперь вы можете ввести любой текст на английском языке и послушать результат.
Параметр `SPEED` позволяет управлять скоростью речи:
*   `1.0` - нормальная скорость
*   `0.8` - быстрая речь
*   `1.2` - медленная речь

In [26]:
USER_TEXT = "The quick brown fox jumps over the lazy dog."
SPEED = 1.0

audio = synthesize_speech(USER_TEXT, alpha=SPEED)

Text: 'The quick brown fox jumps over the lazy dog.'


Сгененируем несколько предложений сразу:

In [27]:
sentences = [
    "Artificial intelligence is transforming the world.",
    "Deep learning has enabled amazing breakthroughs.",
    "Text to speech synthesis is one of them."
]

for i, sentence in enumerate(sentences, 1):
    print(f"\n[{i}/{len(sentences)}]")
    synthesize_speech(sentence)


[1/3]
Text: 'Artificial intelligence is transforming the world.'



[2/3]
Text: 'Deep learning has enabled amazing breakthroughs.'



[3/3]
Text: 'Text to speech synthesis is one of them.'


### Демонстрация скорости речи

Послушаем, как одна и та же фраза звучит с разной скоростью.
В отличие от ускорения аудиозаписи, FastSpeech меняет именно длительность фонем, не искажая высоту голоса.

In [31]:
text = "I can speak fast and I can speak slow."
speeds = [0.8, 1.0, 1.3]

for alpha in speeds:
    print(f"\nAlpha = {alpha}")
    synthesize_speech(text, alpha=alpha)


Alpha = 0.8
Text: 'I can speak fast and I can speak slow.'



Alpha = 1.0
Text: 'I can speak fast and I can speak slow.'



Alpha = 1.3
Text: 'I can speak fast and I can speak slow.'


<a name="mfa-intro"></a>
### Montreal Forced Alignment (MFA)

**Forced Alignment** (вынужденное выравнивание) — это процесс автоматической синхронизации аудиозаписи речи с её текстовой расшифровкой.

FastSpeech — это **неавторегрессионная** модель (Non-Autoregressive).
*   Авторегрессионные модели (Tacotron 2) генерируют речь последовательно и сами решают, сколько времени уделить каждой фонеме с помощью механизма внимания (Attention).
*   FastSpeech генерирует всю мел-спектрограмму параллельно. Для этого модели нужно **заранее знать длительность** каждой фонемы.

<center><img src="img/mfa.png" width=450></center>

<a name="mfa-install"></a>
### Установка и скачивание моделей

MFA работает с любым языком, для которого есть:
1.  **Акустическая модель** (Acoustic Model): научилась понимать звуки этого языка.
2.  **Словарь произношений** (Dictionary/Lexicon): знает, из каких фонем состоят слова (например, "hello" -> "HH AH0 L OW1").

Мы будем использовать официальные **предобученные модели для русского языка** (`russian_mfa`).

In [ ]:
# !conda install -c conda-forge montreal-forced-aligner -y
# !mfa model download acoustic russian_mfa
# !mfa model download dictionary russian_mfa

<a name="mfa-record"></a>
### Генерация

Для работы MFA нужны пары файлов: аудио (`.wav`) и текст (`.lab` или `.txt`).

In [32]:
import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import sounddevice as sd

dataset_dir = "mfa_test_dataset"
os.makedirs(dataset_dir, exist_ok=True)
wav_path = os.path.join(dataset_dir, "test_01.wav")
lab_path = os.path.join(dataset_dir, "test_01.lab")

SR = 22050        
DURATION = 5.0    

def record_and_save():
    print(f"ПРИГОТОВЬТЕСЬ! Запись начнется сразу и продлится {DURATION} секунд.")
    print("Говорите четко и ясно в микрофон...")
    
    recording = sd.rec(int(DURATION * SR), samplerate=SR, channels=1)
    sd.wait()
    print("Запись завершена!")
    
    recording = recording.flatten()
    sf.write(wav_path, recording, SR)
    
    print("\nПрослушайте вашу запись:")
    display(Audio(wav_path))

try:
    record_and_save()
    
    text = input("\n Введите текст, который вы произнесли (на русском, без цифр): ")
    
    if text.strip():
        with open(lab_path, "w") as f:
            f.write(text.lower())
        print(f"\n Успешно сохранено!")
        print(f"Audio: {wav_path}")
        print(f"Text file: {lab_path} -> '{text}'")
    else:
        print("\n Текст не введен! Повторите запуск ячейки.")
        
except Exception as e:
    print(f"\nОшибка при записи: {e}")

ПРИГОТОВЬТЕСЬ! Запись начнется сразу и продлится 5.0 секунд.
Говорите четко и ясно в микрофон...
Запись завершена!

Прослушайте вашу запись:



 Успешно сохранено!
Audio: mfa_test_dataset/test_01.wav
Text file: mfa_test_dataset/test_01.lab -> 'раз два три четыре пят ьвышел зайчик погуля'


<a name="mfa-align"></a>
### Запуск выравнивания

Аргументы команды:
1.  Папка с датасетом (`mfa_test_dataset`)
2.  Словарь (`russian_mfa` — который мы скачали)
3.  Акустическая модель (`russian_mfa` — которую мы скачали)
4.  Папка для сохранения результатов (`mfa_output`)

In [33]:
# Запуск выравнивания
# --clean очищает временные файлы от прошлых запусков
# --single_speaker указывает, что у нас один диктор (для упрощения)

!mfa align mfa_test_dataset russian_mfa russian_mfa mfa_output --clean --single_speaker

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
   1% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 INFO     Found 1 speaker across 1 file, average number of utterances per       
          speaker: 1.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1           
          utterances, MFA will only use 1 jobs.                                 
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating MFCCs...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Calculating CMVN..

<a name="mfa-results"></a>
### Результаты MFA

После выравнивания MFA создает файлы формата **TextGrid**. В них обычно есть два уровня (tiers):

1.  **words (слова)**: показывает, где начинается и заканчивается каждое *слово*.
2.  **phones (фонемы)**: показывает разбивку по *звукам*.


In [ ]:
# !pip install textgrid

In [34]:
import textgrid
import os
import glob

output_dir = "mfa_output"
target_file = "test_01.TextGrid"

found_files = glob.glob(os.path.join(output_dir, "**", target_file), recursive=True)

if not found_files:
    print(f" Файл {target_file} не найден в папке {output_dir}!")
    print("Проверьте вывод ячейки с 'mfa align' на наличие ошибок.")
    if os.path.exists(output_dir):
        print(f"\nСодержимое {output_dir}:")
        for root, dirs, files in os.walk(output_dir):
            for file in files:
                print(os.path.join(root, file))
    else:
        print(f"Папка {output_dir} даже не существует.")
else:
    tg_path = found_files[0]
    print(f" Файл найден: {tg_path}")

    try:
        tg = textgrid.TextGrid.fromFile(tg_path)
        
        phone_tier = tg.getFirst('phones') 
        if not phone_tier:
             phone_tier = tg[1]

        print(f"Используем слой: {phone_tier.name}")

        SR = 22050
        HOP_LENGTH = 256

        print(f"\n{'PHONE':<10} | {'DUR (sec)':<10} | {'DUR (frames)':<10}")
        print("-" * 45)

        total_frames = 0
        
        for interval in phone_tier:
            phone = interval.mark
            
            if phone == "": 
                continue

            duration_sec = interval.maxTime - interval.minTime
            
            duration_frames = int(duration_sec * SR / HOP_LENGTH)
            
            if duration_frames == 0:
                duration_frames = 1
                
            print(f"{phone:<10} | {duration_sec:<10.4f} | {duration_frames:<10d}")
            total_frames += duration_frames
            
        print("-" * 45)
        print(f"Всего фреймов: {total_frames}")
        print(f"Длительность аудио: {total_frames * HOP_LENGTH / SR:.2f} сек")
        
    except Exception as e:
        print(f"Ошибка при чтении TextGrid: {e}")

 Файл найден: mfa_output/test_01.TextGrid
Используем слой: phones

PHONE      | DUR (sec)  | DUR (frames)
---------------------------------------------
r          | 0.1300     | 11        
a          | 0.1700     | 14        
s̪         | 0.0900     | 7         
d̪         | 0.1000     | 8         
v          | 0.1100     | 9         
a          | 0.3900     | 33        
t̪         | 0.0500     | 4         
rʲ         | 0.0800     | 6         
i          | 0.2200     | 18        
tɕ         | 0.1600     | 13        
ɪ          | 0.0300     | 2         
t̪         | 0.1000     | 8         
ɨ          | 0.1300     | 11        
rʲ         | 0.0700     | 6         
e          | 0.0700     | 6         
pʲ         | 0.1800     | 15        
æ          | 0.1300     | 11        
tʲ         | 0.3100     | 26        
v          | 0.2700     | 23        
ɨ          | 0.0500     | 4         
ʂ          | 0.1600     | 13        
ɨ          | 0.0400     | 3         
ɫ          | 0.0600     | 5       